In [1]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [3]:
#data = pd.read_stata(r"Z:\survey\ECU\ENEMDU\1991\m11\data_orig\ECU_1991m11.dta") # para bases de stata
data = pd.read_stata(r"datos/ECU_1991m11_BID.dta") # para bases de stata

## Revisar los datos

- rn - región natural
- estrato - estrato
- cuidad - ciudad
- zona - zona
- sector - sector
- vivienda - vivienda
- hogar - hogar
- persona - persona
- numpers - número de personas
- edad - edad
- ingobr - Ingresos como obrero o empleado
- ingpat - Ingresos como patrono o cuenta propia
- ingalq - Ingresos por alquileres, rentas o interese
- ingjub - Ingresos por jubilación o pensión
- ingotr - por otros ingresos
- fexp - factor de expansión
- ingrl - ingresos

El valor de 'ingobr' es el ingreso laboral monetario, no hay datos sobre ingreso laboral no monetario, las otras variables son ingreso no laboral monetario y no monetario e ingrl es un ingreso total

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no ocupado, se puede usar estas variables y el ingreso laboral asumiendo que cuando estaba ocupado tenía ese ingreso para intentar aproximar el salario mensual y de ahí el salario trimestral, esto solo funciona así ya que no tenemos una variable que explicite el mes, en encuestas que tengan el mes o trimestre explícito esto no sería igual.

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39071 entries, 0 to 39070
Columns: 240 entries, region_BID_c to ppp_wdi2011
dtypes: category(74), float32(16), float64(104), int16(8), int32(1), int8(28), object(9)
memory usage: 40.6+ MB


Filtramos solo las columnas de interés para alivar el peso en la memoria

In [5]:
data.columns

Index(['region_BID_c', 'region_c', 'pais_c', 'anio_c', 'mes_c', 'zona_c',
       'factor_ch', 'idh_ch', 'idp_ci', 'factor_ci',
       ...
       'ingrl', 'peamsiu', 'fexp', 'contrato_ci', 'segsoc_ci', 'formal',
       'formal_1', 'ylm_ci1', 'ynlm_ci1', 'ppp_wdi2011'],
      dtype='object', length=240)

In [6]:
data = data[['rn', 'estrato', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar', 'persona', 'numpers', 'edad', 'ingobr', 'ingpat', 'ingalq',
      'ingjub', 'ingotr', 'fexp', 'ingrl', 'ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'nov', 'dic']]

Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingobr' siempre que reportan estar ocupados en un mes

In [7]:
data['ingr_ene'] = data.apply(lambda x: x['ingobr'] if x['ene'] == 'ocupado' else None, axis=1)
data['ingr_feb'] = data.apply(lambda x: x['ingobr'] if x['feb'] == 'ocupado' else None, axis=1)
data['ingr_mar'] = data.apply(lambda x: x['ingobr'] if x['mar'] == 'ocupado' else None, axis=1)
data['ingr_abr'] = data.apply(lambda x: x['ingobr'] if x['abr'] == 'ocupado' else None, axis=1)
data['ingr_may'] = data.apply(lambda x: x['ingobr'] if x['may'] == 'ocupado' else None, axis=1)
data['ingr_jun'] = data.apply(lambda x: x['ingobr'] if x['jun'] == 'ocupado' else None, axis=1)
data['ingr_jul'] = data.apply(lambda x: x['ingobr'] if x['jul'] == 'ocupado' else None, axis=1)
data['ingr_ago'] = data.apply(lambda x: x['ingobr'] if x['ago'] == 'ocupado' else None, axis=1)
data['ingr_sep'] = data.apply(lambda x: x['ingobr'] if x['sep'] == 'ocupado' else None, axis=1)
data['ingr_oct'] = data.apply(lambda x: x['ingobr'] if x['oct'] == 'ocupado' else None, axis=1)
data['ingr_nov'] = data.apply(lambda x: x['ingobr'] if x['nov'] == 'ocupado' else None, axis=1)
data['ingr_dic'] = data.apply(lambda x: x['ingobr'] if x['dic'] == 'ocupado' else None, axis=1)

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014, utilizamos el IPC de Estados Unidos para ajustar por inflación ya que no podemos usar la inflación en sucres si queremos dejar el valor final en dólares de 2014

In [8]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 1991]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc y tipo de cambio

In [9]:
ipc_dict = dict(zip(datos_actual['trimestre'], datos_actual['IPC Estados Unidos']))

ipc_base_dict = dict(zip(datos_base['trimestre'], datos_base['IPC Estados Unidos']))

tipo_cambio_dict = dict(zip(datos_actual['trimestre'], datos_actual['tipo de cambio']))

### Asignamos el ipc y tipo de cambio correspondiente según trimestre

$\begin{equation}
    ingr_{USD-base-2014}^{i} = \frac{ingr_{sucres}^{i}}{tipo-de-cambio^{i}}\left( \frac{ipcUSA^{i}_{2014}}{ipcUSA^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

In [10]:
# Función que asigna valores correspondientes
def asigna_ipc(trimestre):
    return ipc_dict.get(trimestre, {})

def asigna_ipc_base(trimestre):
    return ipc_base_dict.get(trimestre, {})

In [11]:
data['ipc_t1'] = ipc_dict.get(1)
data['ipc_base_t1'] = ipc_base_dict.get(1)
data['tipo_cambio_t1'] = tipo_cambio_dict.get(1)

data['ipc_t2'] = ipc_dict.get(2)
data['ipc_base_t2'] = ipc_base_dict.get(2)
data['tipo_cambio_t2'] = tipo_cambio_dict.get(2)

data['ipc_t3'] = ipc_dict.get(3)
data['ipc_base_t3'] = ipc_base_dict.get(3)
data['tipo_cambio_t3'] = tipo_cambio_dict.get(3)

data['ipc_t4'] = ipc_dict.get(4)
data['ipc_base_t4'] = ipc_base_dict.get(4)
data['tipo_cambio_t4'] = tipo_cambio_dict.get(4)

In [12]:
# Calculamos el deflactor
data['def_t1'] = (data['ipc_base_t1'] / data['ipc_t1'])
data['def_t2'] = (data['ipc_base_t2'] / data['ipc_t2'])
data['def_t3'] = (data['ipc_base_t3'] / data['ipc_t3'])
data['def_t4'] = (data['ipc_base_t4'] / data['ipc_t4'])

In [13]:
# Ingreso real por mes
data['ingr_ene_r'] = (data['ingr_ene'] / data['tipo_cambio_t1']) * data['def_t1']
data['ingr_feb_r'] = (data['ingr_feb'] / data['tipo_cambio_t1']) * data['def_t1']
data['ingr_mar_r'] = (data['ingr_mar'] / data['tipo_cambio_t1']) * data['def_t1']
data['ingr_abr_r'] = (data['ingr_abr'] / data['tipo_cambio_t2']) * data['def_t2']
data['ingr_may_r'] = (data['ingr_may'] / data['tipo_cambio_t2']) * data['def_t2']
data['ingr_jun_r'] = (data['ingr_jun'] / data['tipo_cambio_t2']) * data['def_t2']
data['ingr_jul_r'] = (data['ingr_jul'] / data['tipo_cambio_t3']) * data['def_t3']
data['ingr_ago_r'] = (data['ingr_ago'] / data['tipo_cambio_t3']) * data['def_t3']
data['ingr_sep_r'] = (data['ingr_sep'] / data['tipo_cambio_t3']) * data['def_t3']
data['ingr_oct_r'] = (data['ingr_oct'] / data['tipo_cambio_t4']) * data['def_t4']
data['ingr_nov_r'] = (data['ingr_nov'] / data['tipo_cambio_t4']) * data['def_t4']
data['ingr_dic_r'] = (data['ingr_dic'] / data['tipo_cambio_t4']) * data['def_t4']

Ingreso mensual promedio en el trimeste

In [14]:
data['ingr_t1_r'] = (data['ingr_ene_r'] + data['ingr_feb_r'] + data['ingr_mar_r'])/3
data['ingr_t2_r'] = (data['ingr_abr_r'] + data['ingr_may_r'] + data['ingr_jun_r'])/3
data['ingr_t3_r'] = (data['ingr_jul_r'] + data['ingr_ago_r'] + data['ingr_sep_r'])/3
data['ingr_t4_r'] = (data['ingr_oct_r'] + data['ingr_nov_r'] + data['ingr_dic_r'])/3

## Calculo ingreso de los hogares

In [15]:
columnas_idef = ['rn', 'estrato', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar']

data['idef_hogar'] = data[columnas_idef].astype(str).agg(''.join, axis=1)
len(data['idef_hogar'].unique())

8243

In [16]:
data[['rn', 'estrato', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar', 'idef_hogar', 'persona', 'numpers']]

,rn,estrato,ciudad,zona,sector,vivienda,hogar,idef_hogar,persona,numpers
0,1,3,10150,001,005,01,1,1310150001005011,3,3
1,1,3,10150,001,005,01,1,1310150001005011,1,3
2,1,3,10150,001,005,01,1,1310150001005011,2,3
3,1,3,10150,001,005,02,1,1310150001005021,1,2
4,1,3,10150,001,005,02,1,1310150001005021,2,2
...,...,...,...,...,...,...,...,...,...,...
39066,3,0,210450,001,011,11,1,30210450001011111,3,5
39067,3,0,210450,001,011,12,1,30210450001011121,3,4
39068,3,0,210450,001,011,12,1,30210450001011121,2,4
39069,3,0,210450,001,011,12,1,30210450001011121,1,4


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [17]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [18]:
data['ingr_t1_h'] = data.groupby('idef_hogar')['ingr_t1_r'].transform(sum_with_na)
data['ingr_t2_h'] = data.groupby('idef_hogar')['ingr_t2_r'].transform(sum_with_na)
data['ingr_t3_h'] = data.groupby('idef_hogar')['ingr_t3_r'].transform(sum_with_na)
data['ingr_t4_h'] = data.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)

In [20]:
data[['ingr_t1_h', 'ingr_t2_h', 'ingr_t3_h', 'ingr_t4_h']].mean()

ingr_t1_h    295.581289
ingr_t2_h    270.916819
ingr_t3_h    264.935781
ingr_t4_h    232.769974
dtype: object

## Sacamos edades negativas y mayores a 100 años

In [22]:
len(data)

39071

En este caso en la variable edad tenemos números y el texto 'menos de un año' así que primero transformamos todas las filas que digan 'menos de un año' a 0

In [23]:
data['edad'] = data['edad'].apply(lambda x: x if type(x) == int else 0)

In [24]:
data = data.loc[(data['edad'] >= 0) & (data['edad'] < 100)]
len(data)

39071

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [25]:
k = 0.4
s = 0.9

In [26]:
# Si es necesario calcular el número de niños
data['es_nino'] = data['edad'] < 10

data['ninos'] = data.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data['es_adulto'] = data['edad'] > 10

data['adultos'] = data.groupby('idef_hogar')['es_adulto'].transform('sum')

In [27]:
data['escala'] = (data['adultos'] + k * data['ninos']) ** s

In [28]:
data['ingr_t_t1'] = data['ingr_t1_h'] / data['escala']
data['ingr_t_t2'] = data['ingr_t2_h'] / data['escala']
data['ingr_t_t3'] = data['ingr_t3_h'] / data['escala']
data['ingr_t_t4'] = data['ingr_t4_h'] / data['escala']

In [29]:
data[['ingr_t_t1', 'ingr_t_t2', 'ingr_t_t3', 'ingr_t_t4']]

,ingr_t_t1,ingr_t_t2,ingr_t_t3,ingr_t_t4
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN
...,...,...,...,...
39066,9.942692,9.093862,8.851951,7.812702
39067,54.824632,50.144132,48.810219,43.079731
39068,54.824632,50.144132,48.810219,43.079731
39069,54.824632,50.144132,48.810219,43.079731


In [30]:
print("Ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].mean())
print("Mediana del ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].median())

Ingreso individual descontando cargas familiares t4:  60.07216153251704
Mediana del ingreso individual descontando cargas familiares t4:  41.74446318843047


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [31]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

In [32]:
datos_final = pd.DataFrame(index=['t1', 't2', 't3', 't4'], columns=['fgt0', 'fgt1', 'fgt2', 'a25', 'a50', 'a75', 'ingreso_promedio'])

In [33]:
data['persona_fexp'] = 1 * data['fexp']

In [34]:
for t in [1, 2, 3, 4]:
    col_ingr = f'ingr_t_t{t}'
    col_pobres = f'pobres_t{t}'

    # una columna que identifica a quienes están por debajo de la línea de pobreza por trimestre
    data[col_pobres] = (
        (data[col_ingr] - umbral_dict.get(t)) < 0
    ).astype(int)

In [35]:
data[['pobres_t1', 'pobres_t2', 'pobres_t3', 'pobres_t4']].sum()

pobres_t1    17198
pobres_t2    18410
pobres_t3    18883
pobres_t4    20322
dtype: int64

In [36]:
print("pobreza t1: ", (data['pobres_t1'] * data['fexp']).sum()/data.loc[data['ingr_t_t1'] >= 0]['persona_fexp'].sum())
print("pobreza t2: ", (data['pobres_t2'] * data['fexp']).sum()/data.loc[data['ingr_t_t2'] >= 0]['persona_fexp'].sum())
print("pobreza t3: ", (data['pobres_t3'] * data['fexp']).sum()/data.loc[data['ingr_t_t3'] >= 0]['persona_fexp'].sum())
print("pobreza t4: ", (data['pobres_t4'] * data['fexp']).sum()/data.loc[data['ingr_t_t4'] >= 0]['persona_fexp'].sum())

pobreza t1:  0.6110452282962818
pobreza t2:  0.6477199541889531
pobreza t3:  0.6585181470744279
pobreza t4:  0.7151960100237409


In [37]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = data.loc[data[f'ingr_t_t{t}'] >= 0].copy()

    # Calculamos una columna de pobres
    df_temp['pobres'] = (df_temp[f'ingr_t_t{t}'] - umbral_dict[t]) < 0

    # Ratio de pobres sobre el total
    ratio = (umbral_dict[t] - df_temp[f'ingr_t_t{t}']) / umbral_dict[t]

    # Calculamos el índice para alpha 0, 1 y 2 solo donde 'pobres' == True.
    for i in range(3):
        col = f'fgt{i}'
        df_temp[col] = np.where(df_temp['pobres'], ratio**i, 0)

    # Cálculo del índice ponderado: se usa el factor de expansión como peso
    peso_total = df_temp['fexp'].sum()
    fgt0 = (df_temp['fgt0'] * df_temp['fexp']).sum() / peso_total
    fgt1 = (df_temp['fgt1'] * df_temp['fexp']).sum() / peso_total
    fgt2 = (df_temp['fgt2'] * df_temp['fexp']).sum() / peso_total
    
    # Guardamos los resultados
    datos_final.loc[f't{t}', 'fgt0'] = fgt0
    datos_final.loc[f't{t}', 'fgt1'] = fgt1
    datos_final.loc[f't{t}', 'fgt2'] = fgt2

In [38]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.611045,0.277652,0.164679,NaN,NaN,NaN,NaN
t2,0.64772,0.307526,0.185074,NaN,NaN,NaN,NaN
t3,0.658518,0.315124,0.190867,NaN,NaN,NaN,NaN
t4,0.715196,0.361614,0.225325,NaN,NaN,NaN,NaN


## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [39]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = data.loc[data[f'ingr_t_t{t}'] >= 0].copy()

    # Suma total de los factores de expansión para el trimestre
    peso_total = df_temp['fexp'].sum()
    
    # Ingreso promedio ponderado
    mu = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total

    # Calculamos el índice A para epsilon 0.25, 0.5 y 0.75 utilizando los pesos
    indices = {}
    for i in [0.25, 0.5, 0.75]:
        A_i = ((df_temp[f'ingr_t_t{t}']**(1-i) * df_temp['fexp']).sum() / peso_total)**(1/(1-i))
        indices[i] = A_i

    # Ratio de pobreza con el índice total (aplicando la fórmula)
    a25 = 1 - 1/mu * indices[0.25]
    a50 = 1 - 1/mu * indices[0.5]
    a75 = 1 - 1/mu * indices[0.75]

    # Guardamos los resultados
    datos_final.loc[f't{t}', 'a25'] = a25
    datos_final.loc[f't{t}', 'a50'] = a50
    datos_final.loc[f't{t}', 'a75'] = a75


In [40]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.611045,0.277652,0.164679,0.096551,0.181786,0.2579,NaN
t2,0.64772,0.307526,0.185074,0.096108,0.180955,0.256714,NaN
t3,0.658518,0.315124,0.190867,0.095843,0.180667,0.25668,NaN
t4,0.715196,0.361614,0.225325,0.096353,0.18139,0.257259,NaN


Guardamos el ingreso promedio

In [41]:
for t in [1, 2, 3, 4]:
    df_temp = data.loc[data[f'ingr_t_t{t}'] >= 0].copy()
    
    # Calcula la suma total de los factores de expansión
    peso_total = df_temp['fexp'].sum()
    
    # Calcula el ingreso promedio ponderado
    media_ponderada = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total
    
    datos_final.loc[f't{t}', 'ingreso_promedio'] = media_ponderada

In [42]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.611045,0.277652,0.164679,0.096551,0.181786,0.2579,78.624141
t2,0.64772,0.307526,0.185074,0.096108,0.180955,0.256714,72.002958
t3,0.658518,0.315124,0.190867,0.095843,0.180667,0.25668,70.209365
t4,0.715196,0.361614,0.225325,0.096353,0.18139,0.257259,61.952106


### Inserta los cálculos en la base final

In [43]:
indices = pd.read_csv("indices.csv", encoding='latin-1')

In [44]:
ano = 1991
# Asegurar que el índice de datos_final coincide con trimestres 1..4
datos_final = datos_final.copy()
datos_final["trimestre"] = [1, 2, 3, 4]
datos_final["Año"] = ano

# Reemplazar en indices usando mask
for col in ["fgt0","fgt1","fgt2","a25","a50","a75","ingreso_promedio"]:
    indices.loc[indices["Año"].eq(ano), col] = datos_final[col].values

/tmp/ipykernel_94254/1450961308.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.6110452282962818 0.6477199541889531 0.6585181470744279
 0.7151960100237409]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  indices.loc[indices["Año"].eq(ano), col] = datos_final[col].values
/tmp/ipykernel_94254/1450961308.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.27765219260879387 0.3075255004443378 0.3151241868603877
 0.3616142995843563]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  indices.loc[indices["Año"].eq(ano), col] = datos_final[col].values
/tmp/ipykernel_94254/1450961308.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.16467912652262257 0.18507418999026232 0.190866801078

In [46]:
indices.to_csv('indices.csv', encoding='latin-1', index=None)